In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## Custom Guardrails

### Before Agent

In [2]:
# 차단 및 제재 키워드 정의
forbidden_topics = {
    "cheating": ["답지", "정답 알려줘", "숙제 대신", "써줘", "베끼기"], # 부정행위 관련
    "distraction": ["롤", "게임", "유튜브", "아이돌", "웹툰", "웃긴"], # 학습 방해 요소
    "harmful": ["담배", "술", "폭력", "싸움", "바보"] # 유해 콘텐츠
}

In [4]:
from langchain.agents.middleware import before_agent

@before_agent(can_jump_to=["end"])
def education_guardrail(state, runtime):
    """
    학생의 질문 의도를 파악하여 교육적이지 않거나 부정행위가 의심될 경우,
    LLM(AI)에게 질문을 넘기지 않고 교육적인 멘트로 즉시 교정합니다.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]:
        return None

    last_message = state["messages"][-1]
    if last_message.type != "human":
        return None

    user_text = last_message.content

    # 2. 카테고리별 검사 로직
    # 단순히 막는 것을 넘어, '왜' 안되는지 카테고리별로 다른 피드백 주기

    # Case A: 부정행위 방지 (Cheating Prevention)
    # AI가 숙제를 통째로 해주는 것을 방지
    for keyword in forbidden_topics["cheating"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "🚫 스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 드릴까요? 어떤 부분이 가장 어려운지 말해주세요."
                }],
                "jump_to": "end"
            }

    # Case B: 학습 집중 유도 (Focus Management)
    # 공부 중에 게임이나 딴짓 이야기를 하면 다시 공부로 유도
    for keyword in forbidden_topics["distraction"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?"
                }],
                "jump_to": "end"
            }

    # Case C: 유해 콘텐츠 차단 (Safety)
    # 교육 서비스의 브랜드 안전성(Brand Safety)을 위한 기능
    for keyword in forbidden_topics["harmful"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⚠️ 부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요."
                }],
                "jump_to": "end"
            }

    # 3. 통과 (Pass)
    # 위 조건들에 걸리지 않으면 정상적으로 AI 튜터(LLM)가 답변 생성
    return None

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    tools=[],
    middleware=[education_guardrail],
)

In [6]:
agent.invoke({
    "messages": [{"role": "user", "content": "피타고라스의 정리가 이해가 안 돼. 설명해줘."}]
})

{'messages': [HumanMessage(content='피타고라스의 정리가 이해가 안 돼. 설명해줘.', additional_kwargs={}, response_metadata={}, id='a253f5fc-a2ec-4c56-82f6-4db74347d26b'),
  AIMessage(content=[{'type': 'text', 'text': '피타고라스의 정리는 처음 들으면 조금 복잡해 보일 수 있지만, 사실 **\'직각삼각형의 변들 사이의 아주 특별한 관계\'**를 말하는 거예요.\n\n가장 쉽게 이해할 수 있도록 단계별로 설명해 드릴게요.\n\n---\n\n### 1. 피타고라스의 정리가 뭐야?\n직각삼각형에서 **가장 긴 변(빗변)을 제곱한 값은, 나머지 두 변을 각각 제곱해서 더한 값과 같다**는 법칙이에요.\n\n공식으로 쓰면 이렇습니다:\n> **$a^2 + b^2 = c^2$**\n> *(여기서 $a, b$는 짧은 두 변, $c$는 가장 긴 빗변이에요.)*\n\n---\n\n### 2. 그림으로 이해하기 (핵심!)\n숫자로만 보면 어렵지만, **\'정사각형\'**을 상상하면 아주 쉬워요.\n\n1. 직각삼각형이 하나 있다고 해볼게요. 세 변의 길이를 각각 $a, b, c$라고 합시다.\n2. 각 변을 한 변으로 하는 **정사각형**을 하나씩 그려보세요.\n3. 그러면 삼각형 주위에 정사각형이 3개 생기죠?\n4. **"작은 두 정사각형의 넓이를 합치면, 가장 큰 정사각형의 넓이와 같다"**는 게 바로 피타고라스 정리예요.\n\n*   작은 정사각형의 넓이 = $a \\times a = a^2$\n*   중간 정사각형의 넓이 = $b \\times b = b^2$\n*   큰 정사각형의 넓이 = $c \\times c = c^2$\n*   즉, **$a^2 + b^2 = c^2$**\n\n---\n\n### 3. 숫자로 예를 들어볼까요? (가장 유명한 3:4:5)\n밑변($a$)이 **3cm**, 높이($b$)가 **4cm**인 직각삼각형

In [7]:
agent.invoke({
    "messages": [{"role": "user", "content": "독후감 대신 써줘."}]
})

{'messages': [HumanMessage(content='독후감 대신 써줘.', additional_kwargs={}, response_metadata={}, id='d9029c90-2b38-4f1e-93c1-2f59db097dc6'),
  AIMessage(content='🚫 스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 드릴까요? 어떤 부분이 가장 어려운지 말해주세요.', additional_kwargs={}, response_metadata={}, id='4435e579-7826-49c8-9460-ba4c48652e09', tool_calls=[], invalid_tool_calls=[])]}

In [8]:
agent.invoke({
    "messages": [{"role": "user", "content": "웃긴 얘기해줘"}]
})

{'messages': [HumanMessage(content='웃긴 얘기해줘', additional_kwargs={}, response_metadata={}, id='5cbf5bcc-2445-484f-aca4-b745b68e2807'),
  AIMessage(content='⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?', additional_kwargs={}, response_metadata={}, id='f98bdbe7-3c7f-4ef9-ab7c-db742045434f', tool_calls=[], invalid_tool_calls=[])]}

In [9]:
agent.invoke({
    "messages": [{"role": "user", "content": "재밌는 유튜브 알려줘"}]
})

{'messages': [HumanMessage(content='재밌는 유튜브 알려줘', additional_kwargs={}, response_metadata={}, id='aaf5a12d-4e51-4dbc-afc8-659c924e6c66'),
  AIMessage(content='⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?', additional_kwargs={}, response_metadata={}, id='9fc84f3f-56eb-4d5d-affd-de77425a4423', tool_calls=[], invalid_tool_calls=[])]}

In [10]:
agent.invoke({
    "messages": [{"role": "user", "content": "담배 피면 좋아?"}]
})

{'messages': [HumanMessage(content='담배 피면 좋아?', additional_kwargs={}, response_metadata={}, id='feaca9ea-fdde-41ce-b342-8a2875b9a91b'),
  AIMessage(content='⚠️ 부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요.', additional_kwargs={}, response_metadata={}, id='7e777593-5c54-4f38-9f73-2faa10cc99ec', tool_calls=[], invalid_tool_calls=[])]}

### After Agent

In [26]:
from langchain.chat_models import init_chat_model

safety_model = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [27]:
from langchain.agents.middleware import after_agent
from langchain.messages import AIMessage

@after_agent
def answer_leakage_guardrail(state, runtime) :
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 마지막 메시지가 AI의 답변이 아니면 검사할 필요 없음
    if not isinstance(last_message, AIMessage):
        return None

    # 2. 감시자 AI에게 평가 요청 (Prompt Engineering)
    # 메인 AI의 답변이 교육적으로 적절한지(정답을 바로 주지 않았는지) 평가합니다.
    auditor_prompt = f"""
    당신은 엄격한 교육 감독관입니다.
    다음 '튜터의 답변'을 확인하세요.
    답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
    답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

    튜터의 답변: {last_message.content}
    """

    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])

    # 3. 결과에 따른 개입 (Intervention)
    if "LEAKED" in result.content:
        print(f"🚨 [가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.")
        last_message.content = "앗, 제가 정답을 바로 말할 뻔했네요! 😅 정답보다는 푸는 방법을 먼저 생각해볼까요? 이 문제의 핵심 개념은..."

    return None

In [28]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    tools=[],
    middleware=[answer_leakage_guardrail],
)

In [29]:
agent.invoke({
    "messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘."}]
})

🚨 [가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.


{'messages': [HumanMessage(content='직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘.', additional_kwargs={}, response_metadata={}, id='6200835e-45d0-4bce-a106-13e8c0b715ba'),
  AIMessage(content='앗, 제가 정답을 바로 말할 뻔했네요! 😅 정답보다는 푸는 방법을 먼저 생각해볼까요? 이 문제의 핵심 개념은...', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7ac8-8bed-74f2-b91f-752ae9577921-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 122, 'total_tokens': 150, 'input_token_details': {'cache_read': 0}})]}

In [30]:
from langchain.agents.middleware import after_agent
from langchain.messages import SystemMessage, AIMessage, HumanMessage

@after_agent
def answer_leakage_guardrail(state, runtime):
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 마지막 메시지가 AI의 답변이 아니면 검사할 필요 없음
    if not isinstance(last_message, AIMessage):
        return None

    # 2. 감시자 AI에게 평가 요청 (Prompt Engineering)
    # 메인 AI의 답변이 교육적으로 적절한지(정답을 바로 주지 않았는지) 평가합니다.
    auditor_prompt = f"""
    당신은 엄격한 교육 감독관입니다.
    다음 '튜터의 답변'을 확인하세요.
    답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
    답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

    튜터의 답변: {last_message.content}
    """

    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])

    # 3단계: 교정 (Correction / Regeneration)
    if "LEAKED" in result.content:

        # 원래 사용자의 질문을 가져오기 (문맥 파악용) -> state["messages"][-2]가 보통 사용자 질문
        original_question = state["messages"][-2].content if len(state["messages"]) >= 2 else "사용자 질문 알 수 없음"

        # 교정 모델에게 "정답을 빼고 힌트로 바꿔라"고 지시
        correction_prompt = f"""
        당신은 친절한 AI 튜터입니다.

        절대 정답을 직접 말하지 말고, 학생이 스스로 생각할 수 있도록 유도하는 질문이나 핵심 개념(힌트)만 설명하세요.
        말투는 친절하게 해주세요.

        사용자 질문: {original_question}
        """

        # LLM을 다시 호출하여 새로운 답변 생성 (비용은 1회 더 발생하지만 품질 확보)
        corrected_response = safety_model.invoke([
            SystemMessage(content="당신은 소크라테스식 교육법을 사용하는 튜터입니다."),
            HumanMessage(content=correction_prompt)
        ])

        # 원래의 유출된 답변을 교정된 답변으로 덮어쓰기
        last_message.content = corrected_response.content

    return None


In [32]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    tools=[],
    middleware=[answer_leakage_guardrail],
)

In [33]:
agent.invoke({
    "messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘."}]
})

{'messages': [HumanMessage(content='직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘.', additional_kwargs={}, response_metadata={}, id='312ac66f-7edb-4923-8521-77a57938d8eb'),
  AIMessage(content='안녕하세요! 직각삼각형에 대한 흥미로운 질문이네요. 😊\n\n직각삼각형의 두 직각변 길이를 알고 있을 때 빗변의 길이를 구하는 데 도움이 되는 아주 유명한 정리가 있어요. 혹시 들어보신 적 있나요?\n\n이 정리가 무엇인지, 그리고 어떻게 사용되는지 함께 이야기 나눠볼까요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7ac9-609e-7483-adc6-df7f917de5a7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 119, 'total_tokens': 147, 'input_token_details': {'cache_read': 0}})]}

### Combine multiple Guardrails

In [34]:
import re

@before_agent
def student_safety_middleware(state, runtime):
    """
    학생의 전화번호나 이메일이 감지되면 마스킹 처리하여 안전을 확보
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]
    if last_message.type != "human": return None

    content = last_message.content
    original_content = content # 로깅용

    # 전화번호 패턴 (010-XXXX-XXXX 또는 010XXXXXXXX 등)
    phone_pattern = r'01[016789]-?[0-9]{3,4}-?[0-9]{4}'
    # 이메일 패턴
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

    is_redacted = False

    if re.search(phone_pattern, content):
        content = re.sub(phone_pattern, '<PHONE_REDACTED>', content)
        is_redacted = True

    if re.search(email_pattern, content):
        content = re.sub(email_pattern, '<EMAIL_REDACTED>', content)
        is_redacted = True

    if is_redacted:
        print(f"🔒 [학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.\n원본: {original_content}\n수정: {content}")
        # 내용을 수정하여 LLM에게 전달 (사용자에게 알릴 필요 없이 조용히 처리하거나, 시스템 메시지 추가 가능)
        last_message.content = content

    return None

In [35]:
ESCALATION_KEYWORDS = ["왕따", "괴롭힘", "우울해", "학교 폭력", "상담 선생님", "사람 불러줘"]

@before_agent(can_jump_to=["end"])
def counseling_escalation_middleware(state, runtime) :
    """
    [Layer 3] 심리적 위기 상황이나 상담 요청이 감지되면 AI 답변을 멈추고 인간 상담사에게 알림을 보냅니다.
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 민감한 키워드가 포함되어 있는지 확인
    for keyword in ESCALATION_KEYWORDS:
        if keyword in last_message.content:
            print(f"✋ [상담 이관] 심각한 고민/요청 감지: {keyword}")

            # 여기서 실제로는 상담 교사에게 알림(Slack, Email 등)을 보내는 로직이 들어감
            # send_alert_to_teacher(last_message.content)

            return {
                "messages": [{
                    "role": "assistant",
                    "content": "학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. \n\n지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)"
                }],
                "jump_to": "end" # AI 답변 생성 중단
            }
    return None


In [36]:
agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[
        education_guardrail,             # Layer 1: 입력 필터 (규칙 - 딴짓/부정행위)
        student_safety_middleware,       # Layer 2: 개인정보 보호 (전화번호 마스킹)
        counseling_escalation_middleware,# Layer 3: 상담 이관 (휴먼 에스컬레이션)
        answer_leakage_guardrail         # Layer 4: 출력 필터 (모델 기반 교정)
    ],
)

In [37]:
agent.invoke({
    "messages": [{"role": "user", "content": "제 번호 010-1234-5678입니다."}]
})

🔒 [학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.
원본: 제 번호 010-1234-5678입니다.
수정: 제 번호 <PHONE_REDACTED>입니다.


{'messages': [HumanMessage(content='제 번호 <PHONE_REDACTED>입니다.', additional_kwargs={}, response_metadata={}, id='e6ec7528-377d-4825-8661-20cf41b04223'),
  AIMessage(content='안녕하세요! 만나서 반갑습니다. 😊\n\n저는 당신이 스스로 답을 찾아갈 수 있도록 돕는 AI 튜터입니다. 궁금한 점이 있다면 무엇이든 편하게 물어보세요.\n\n혹시 무엇에 대해 이야기하고 싶으신가요? 아니면 어떤 것을 배우고 싶으신가요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7aca-954e-7300-a69c-10a9b94af1b3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 54, 'total_tokens': 67, 'input_token_details': {'cache_read': 0}})]}

In [38]:
agent.invoke({
    "messages": [{"role": "user", "content": "요즘 학교에서 왕따 당하고 있어"}]
})

✋ [상담 이관] 심각한 고민/요청 감지: 왕따


{'messages': [HumanMessage(content='요즘 학교에서 왕따 당하고 있어', additional_kwargs={}, response_metadata={}, id='cf157a71-a392-4a48-ba39-258a75d09e1c'),
  AIMessage(content='학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. \n\n지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)', additional_kwargs={}, response_metadata={}, id='54fc8556-bae4-4e29-99d4-5e15cf5a00bd', tool_calls=[], invalid_tool_calls=[])]}